# Two-qubit crosstalk

Qubit $A$ receives an ideal $X$ pulse while an undriven spectator $B$ is coupled through $J\sigma_x^A\sigma_x^B$. This is the first demo where naming the subsystems pays off: the local drive embeds itself, and a partial trace later discards the spectator.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
import htdse as ht

## Write the two pieces of physics

The subsystem dictionary fixes $\mathcal H_A\otimes\mathcal H_B$. A local term needs only `on="A"`; the product term names both factors. No explicit identity matrices are needed.

In [ ]:
Omega, J = 1.0, 0.08
T = np.pi / Omega
subsystems = {"A": 2, "B": 2}

target = (ht.System(subsystems)
          + ht.term(0.5 * Omega * ht.sigma_x, on="A", name="drive"))
crosstalk = ht.term({"A": ht.sigma_x, "B": ht.sigma_x}, coeff=J, name="crosstalk")
realized = target + crosstalk

ht.show(realized)

## Compare the full operations

Both propagators act on the same four-dimensional space, so process fidelity compares them directly.

In [ ]:
U_target = ht.propagator(target, T, verbose=False)
U_realized = ht.propagator(realized, T, verbose=False)
print(f"process fidelity: {ht.process_fidelity(U_target, U_realized):.6f}")

## Ask only what happened to qubit A

Starting from $|00\rangle$, evolve the joint ket, trace out $B$, and compare the resulting mixed state with the intended $|1\rangle_A$. Loss of purity here is entanglement with a modeled subsystem, not dissipation.

In [ ]:
psi0 = ht.ket("00")
psi_T = ht.evolve(realized, psi0, T, verbose=False)
rho_A = ht.partial_trace(psi_T, subsystems, "B")

print(f"fidelity to |1>: {ht.fidelity(ht.ket('1'), rho_A):.6f}")
print(f"purity Tr(rho_A^2): {ht.Tr(rho_A @ rho_A).real:.6f}")

## See the error grow with coupling

In [ ]:
Js = np.linspace(0, 0.2, 21)
fidelities = []
for j in Js:
    H = target + ht.term({"A": ht.sigma_x, "B": ht.sigma_x}, coeff=j)
    psi = ht.evolve(H, psi0, T, verbose=False)
    fidelities.append(ht.fidelity(ht.ket("1"), ht.partial_trace(psi, subsystems, "B")))

plt.plot(Js, fidelities, "o-")
plt.xlabel("crosstalk strength J")
plt.ylabel(r"$F(\rho_A, |1\rangle)$")
plt.ylim(0, 1.02)
plt.show()